# 01 - AgentCore Foundations

This notebook sets up the agent you will use throughout the AgentCore module. You will:

1. See how Runtime, Observability, and Evaluations work together.
2. Meet the `CityAnalyst` agent and inspect its project configuration.
3. Validate and run the agent locally from this notebook.
4. Deploy the agent once and follow one request through its trace.

The remaining notebooks reuse this deployment, so you only need to deploy again if you change the agent code or configuration.

**Estimated time:** 35-50 minutes  
**Creates AWS resources:** Yes

## 1. How the AgentCore pieces fit together

We will use several AgentCore services, each with a clear job:

| Service or tool | Question it answers |
|---|---|
| AgentCore Runtime | Where does my agent code run? |
| AgentCore Observability | What happened during an invocation? |
| AgentCore Evaluations | How well did the session, response, or tool call perform? |
| AgentCore CLI | How do I develop, deploy, inspect, and evaluate the project? |
| Python SDK | How do I automate evaluations and curated scenario runs? |

Observability records an agent request at three useful levels:

```text
session: related turns from one interaction
  +-- trace: one user request and agent response
        +-- span: one model call, tool call, or application step
```

Evaluators use this structure to score the right amount of activity:

- `SESSION`: the complete conversation or task
- `TRACE`: one request and response
- `TOOL_CALL`: one tool execution within a trace

Evaluators work from the recorded inputs, outputs, and tool activity. They do not receive a model's private reasoning.

## 2. Runtime or Harness?

AgentCore can run agent applications in two ways:

- **Runtime** hosts agent code that you write. You choose the framework, tools, and protocol.
- **Harness** provides a managed agent loop that you configure with models, tools, skills, and memory.

This module uses Runtime with the HTTP protocol. Keeping the agent code visible makes it easier to connect each model call and tool call to the trace you will evaluate. The same evaluation concepts also apply to agents built with Harness.

## 3. Prepare the notebook environment

Open this notebook from `Framework Specific Evaluations/AgentCore/`.

Complete the [module prerequisites](README.md#prerequisites) before starting. The next cell installs the Python packages used by all six notebooks. The environment check then confirms the project files, Node.js, the AgentCore CLI, your AWS identity, and your AWS Region.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import subprocess
from pathlib import Path

import boto3

EXPECTED_AGENTCORE_VERSION = "0.27.0"
MODULE_ROOT = Path.cwd()

required_files = (
    MODULE_ROOT / "agentcore" / "agentcore.json",
    MODULE_ROOT / "agentcore" / "cdk" / "package.json",
)
if not all(path.exists() for path in required_files):
    raise RuntimeError("Open this notebook from the AgentCore module directory.")

def command_output(*command):
    return subprocess.run(
        command, check=True, capture_output=True, text=True
    ).stdout.strip()

try:
    node_version = command_output("node", "--version")
    agentcore_version = command_output("agentcore", "--version")
    identity = json.loads(
        command_output("aws", "sts", "get-caller-identity", "--output", "json")
    )
except FileNotFoundError as error:
    raise RuntimeError(
        f"{error.filename} is not installed. Complete the prerequisites in README.md."
    ) from error

if int(node_version.removeprefix("v").split(".")[0]) < 20:
    raise RuntimeError("Node.js 20 or later is required. See README.md.")

if agentcore_version != EXPECTED_AGENTCORE_VERSION:
    raise RuntimeError(
        f"AgentCore CLI {EXPECTED_AGENTCORE_VERSION} is required; found {agentcore_version}. "
        "Install the version listed in README.md."
    )

aws_region = boto3.Session().region_name
if not aws_region:
    raise RuntimeError("Configure an AWS Region before continuing. See README.md.")

print("Node.js:", node_version)
print("AgentCore CLI:", agentcore_version)
print("AWS account:", identity["Account"])
print("AWS Region:", aws_region)

## 4. Inspect the project

Before running the agent, take a quick look at the files that define it:

- `app/CityAnalyst/` contains the Python agent, its prompt, and its city data.
- `agentcore/agentcore.json` tells the CLI how to package and run the agent.
- `agentcore/aws-targets.json` stores the AWS account and Region selected for deployment.
- `agentcore/cdk/` contains the CLI-managed infrastructure used during deployment. You do not need to edit it.
- `agentcore/.cli/deployed-state.json` is created after deployment and records the resulting AWS resources.

The build type is `CodeZip`. This means the CLI packages the Python source and dependencies as a zip file and deploys them without requiring Docker on your computer. A container build is more appropriate when an agent needs custom operating-system packages or a custom image.

The configuration also uses AWS IAM authentication, records model and tool activity with OpenTelemetry, and limits how long an idle session stays active. The next cell shows only the fields that matter for this notebook.

In [ ]:
config = json.loads(
    (MODULE_ROOT / "agentcore" / "agentcore.json").read_text()
)
runtime_config = config["runtimes"][0]
{
    "runtime_name": runtime_config["name"],
    "agent_code": runtime_config["codeLocation"],
    "build_type": runtime_config["build"],
    "protocol": runtime_config["protocol"],
    "network_mode": runtime_config["networkMode"],
    "authentication": runtime_config["authorizerType"],
    "tracing_enabled": runtime_config["instrumentation"]["enableOtel"],
    "idle_session_timeout_seconds": runtime_config["lifecycleConfiguration"]["idleRuntimeSessionTimeout"],
}

## 5. Meet CityAnalyst

`CityAnalyst` is a small city-information assistant built with Strands and an Amazon Bedrock model. A user can ask it to look up a city, compare two cities, or calculate population density. The same agent will be used throughout this module for trace inspection, evaluator design, datasets, and monitoring.

Evaluation works best when expected behavior is clear. Live web results change over time, so CityAnalyst reads from a fixed workshop dataset and exposes three focused tools:

| Tool | Intended use |
|---|---|
| `lookup_city(city, state)` | One named city |
| `compare_cities(city_a, state_a, city_b, state_b)` | A two-city comparison |
| `calculate_density(population, land_area_mi2)` | Math from user-supplied values |

The agent also returns a structured JSON response. The language model may still vary its wording, but the facts, tool behavior, and response shape remain stable. That gives us useful evaluation targets: factual accuracy, tool selection, tool parameters, and schema compliance.

In [ ]:
facts = json.loads(
    (MODULE_ROOT / "app" / "CityAnalyst" / "data" / "city_facts.json").read_text()
)
print(facts["data_version"])
print(f"Cities: {len(facts['cities'])}")
facts["cities"][:3]

In [ ]:
print(
    (MODULE_ROOT / "app" / "CityAnalyst" / "system_prompt.txt").read_text()
)

## 6. Validate and test locally

Validation checks the project configuration before any server or AWS resource is started.

In [ ]:
subprocess.run(["agentcore", "validate"], cwd=MODULE_ROOT, check=True)

Next, run the agent locally. The cell below starts the development server in the background, waits for it to become ready, sends one prompt, and then stops the server. Local mode calls the configured Bedrock model but does not create an AgentCore Runtime in your AWS account.

When AWS credentials are available, local runs can also send trace data to CloudWatch through OpenTelemetry, the tracing standard used by AgentCore. CityAnalyst keeps recent turns in a small local cache that disappears when the process stops. AgentCore Memory is available when an application needs conversation data to persist.

In [ ]:
import socket
import time

DEV_PORT = 8080
LOCAL_PROMPT = "Compare Seattle, WA with Portland, OR. Which city is denser?"
DEV_LOG_PATH = MODULE_ROOT / "generated" / "agentcore-dev.log"
DEV_LOG_PATH.parent.mkdir(exist_ok=True)

def wait_for_local_server(process, port, timeout_seconds=90):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        if process.poll() is not None:
            log_tail = DEV_LOG_PATH.read_text(errors="replace")[-4000:]
            raise RuntimeError(f"Local server stopped during startup:\n{log_tail}")
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return
        except OSError:
            time.sleep(1)
    raise TimeoutError(f"Local server was not ready within {timeout_seconds} seconds.")

# Start the local server and write its logs to generated/agentcore-dev.log.
with DEV_LOG_PATH.open("w") as dev_log:
    dev_process = subprocess.Popen(
        [
            "agentcore",
            "dev",
            "--runtime",
            "CityAnalyst",
            "--port",
            str(DEV_PORT),
            "--logs",
        ],
        cwd=MODULE_ROOT,
        stdout=dev_log,
        stderr=subprocess.STDOUT,
        text=True,
    )
    try:
        # Wait for startup, then send one prompt to the running server.
        wait_for_local_server(dev_process, DEV_PORT)
        subprocess.run(
            [
                "agentcore",
                "dev",
                "--runtime",
                "CityAnalyst",
                "--port",
                str(DEV_PORT),
                "--stream",
                LOCAL_PROMPT,
            ],
            cwd=MODULE_ROOT,
            check=True,
        )
    finally:
        # Stop the local server even if the invocation fails.
        dev_process.terminate()
        try:
            dev_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            dev_process.kill()
            dev_process.wait()

print(f"Local server stopped. Logs: {DEV_LOG_PATH}")

## 7. Enable trace search

AgentCore sends trace data to CloudWatch. CloudWatch Transaction Search makes those traces searchable by the AgentCore CLI and evaluation tools.

This is an account-level setting for the current AWS Region and usually needs to be enabled only once. The next cell checks the current destination and updates it only when necessary. Trace data takes a few seconds to arrive, so later cells check repeatedly for up to two minutes.

In [ ]:
def get_trace_destination():
    result = subprocess.run(
        [
            "aws",
            "xray",
            "get-trace-segment-destination",
            "--output",
            "json",
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(result.stdout)

trace_destination = get_trace_destination()

if trace_destination.get("Destination") == "CloudWatchLogs":
    print("CloudWatch Transaction Search is already enabled.")
else:
    update = subprocess.run(
        [
            "aws",
            "xray",
            "update-trace-segment-destination",
            "--destination",
            "CloudWatchLogs",
        ],
        capture_output=True,
        text=True,
    )
    trace_destination = get_trace_destination()

    if trace_destination.get("Destination") != "CloudWatchLogs":
        raise RuntimeError(
            "Could not enable CloudWatch Transaction Search:\n"
            + update.stderr.strip()
        )
    print("CloudWatch Transaction Search is enabled.")

print(json.dumps(trace_destination, indent=2))

## 8. Deploy the shared runtime

A deployment target gives an AWS account and Region a short name. The first cell below creates a target named `default` when one does not already exist. It uses the AWS account from the environment check and the Region from your AWS environment or CLI configuration. Review the printed values before deploying.

CDK bootstrap prepares an account and Region to receive packaged files and CloudFormation deployments. Complete the one-time bootstrap command in the [module prerequisites](README.md#prerequisites) before the first deployment. The second cell confirms that the selected target is ready.

The final cell deploys in non-interactive mode because a notebook cannot answer terminal prompts. Deployment can take several minutes. If it fails, the cell shows the AgentCore error and the path to the detailed deployment log.

This workshop uses public networking and CLI-generated IAM resources to keep setup short. For a production application, review four choices with your security and platform teams:

- which AWS actions the runtime role is allowed to perform
- whether the runtime should use public networking or a VPC
- how logs are encrypted and how long they are retained
- how callers authenticate before invoking the runtime

Those production decisions are important, but they are not required to complete this evaluation exercise.

In [ ]:
TARGET_NAME = "default"
TARGETS_PATH = MODULE_ROOT / "agentcore" / "aws-targets.json"
targets = json.loads(TARGETS_PATH.read_text())
target = next(
    (item for item in targets if item.get("name") == TARGET_NAME),
    None,
)

if target is None:
    target = {
        "name": TARGET_NAME,
        "description": "AgentCore evaluations workshop",
        "account": identity["Account"],
        "region": aws_region,
    }
    targets.append(target)
    TARGETS_PATH.write_text(json.dumps(targets, indent=2) + "\n")
    print("Created the default deployment target.")
else:
    if target.get("account") != identity["Account"]:
        raise RuntimeError(
            "The default target belongs to AWS account "
            f"{target.get('account')}, but the current identity belongs to "
            f"{identity['Account']}. Update agentcore/aws-targets.json or "
            "sign in to the target account."
        )
    if not target.get("region"):
        raise RuntimeError("The default deployment target does not specify a Region.")
    print("Using the existing default deployment target.")

print(json.dumps(target, indent=2))

In [ ]:
BOOTSTRAP_PARAMETER = "/cdk-bootstrap/hnb659fds/version"
MINIMUM_BOOTSTRAP_VERSION = 8
bootstrap_environment = f"aws://{target['account']}/{target['region']}"

bootstrap_check = subprocess.run(
    [
        "aws", "ssm", "get-parameter",
        "--name", BOOTSTRAP_PARAMETER,
        "--region", target["region"],
        "--query", "Parameter.Value",
        "--output", "text",
    ],
    capture_output=True,
    text=True,
)

if bootstrap_check.returncode != 0:
    raise RuntimeError(
        f"CDK is not ready for {bootstrap_environment}. "
        "Complete the bootstrap step in README.md, then rerun this cell.\n\n"
        + (bootstrap_check.stderr.strip() or bootstrap_check.stdout.strip())
    )

bootstrap_version = int(bootstrap_check.stdout.strip())
if bootstrap_version < MINIMUM_BOOTSTRAP_VERSION:
    raise RuntimeError(
        f"CDK bootstrap version {bootstrap_version} is below the required "
        f"version {MINIMUM_BOOTSTRAP_VERSION}. Rerun the bootstrap step in README.md."
    )

print(f"CDK bootstrap version {bootstrap_version} is ready for {bootstrap_environment}.")

In [ ]:
deploy = subprocess.run(
    [
        "agentcore",
        "deploy",
        "--target",
        TARGET_NAME,
        "--yes",
        "--json",
    ],
    cwd=MODULE_ROOT,
    capture_output=True,
    text=True,
)

try:
    deploy_result = json.loads(deploy.stdout)
except json.JSONDecodeError:
    deploy_result = None

if deploy.returncode != 0:
    error = (
        deploy_result.get("error")
        if isinstance(deploy_result, dict)
        else None
    )
    error = error or deploy.stderr.strip() or deploy.stdout.strip()
    message = f"AgentCore deployment failed: {error or 'No error details returned.'}"

    log_path = (
        deploy_result.get("logPath")
        if isinstance(deploy_result, dict)
        else None
    )
    if log_path:
        message += f"\nDeployment log: {MODULE_ROOT / log_path}"
    raise RuntimeError(message)

print("Deployment completed.")
print(json.dumps(deploy_result, indent=2) if deploy_result else deploy.stdout)

In [ ]:
status = subprocess.run(
    ["agentcore", "status", "--target", TARGET_NAME, "--json"],
    cwd=MODULE_ROOT,
    capture_output=True,
    text=True,
)
if status.returncode != 0:
    raise RuntimeError(
        "Could not read the deployment status:\n"
        + (status.stderr.strip() or status.stdout.strip())
    )

print(json.dumps(json.loads(status.stdout), indent=2))

## 9. Invoke one traceable session

A session ID is the shared key that connects an invocation to its trace and evaluation results. We create one explicitly so later notebooks can reuse this session as evidence.

In [ ]:
from src.workshop_utils import (
    GENERATED_DIR,
    RUNTIME_NAME,
    make_session_id,
    run_cli_json,
)

GENERATED_DIR.mkdir(exist_ok=True)
SESSION_ID = make_session_id("foundations")
PROMPT = "Compare Seattle, WA with Portland, OR. Which city is denser?"

invocation = run_cli_json(
    "invoke",
    "--runtime",
    RUNTIME_NAME,
    "--session-id",
    SESSION_ID,
    "--prompt",
    PROMPT,
)
print(json.dumps(invocation, indent=2))

session_record = {
    "session_id": SESSION_ID,
    "prompt": PROMPT,
}
(GENERATED_DIR / "foundation_session.json").write_text(
    json.dumps(session_record, indent=2) + "\n"
)

## 10. Poll for the trace

The helper checks `agentcore traces list --json` every ten seconds. It returns as soon as the session appears and raises a clear error if two minutes pass without a result.

In [ ]:
from src.workshop_utils import wait_for_session_trace

trace_summary = wait_for_session_trace(SESSION_ID)
trace_summary

In [ ]:
TRACE_ID = trace_summary["traceId"]
trace_path = GENERATED_DIR / "foundation_trace.json"
trace_download = run_cli_json(
    "traces",
    "get",
    TRACE_ID,
    "--runtime",
    RUNTIME_NAME,
    "--output",
    str(trace_path),
)
trace_download

## 11. Read the span hierarchy

A trace is a tree of spans. Reading the parent and child IDs shows the order of model calls, tool calls, and application work.

The cell uses the trace you just downloaded. A small checked-in trace is available as a fallback so you can still inspect the data shape before deploying.

In [ ]:
import pandas as pd
from src.workshop_utils import find_span_records, summarize_spans

source_path = (
    trace_path
    if trace_path.exists()
    else MODULE_ROOT / "data" / "sample_spans.json"
)
trace_document = json.loads(source_path.read_text())
span_rows = summarize_spans(find_span_records(trace_document))
pd.DataFrame(span_rows)

## 12. Foundation checkpoint

At this point, you should be able to explain:

- the roles of Runtime, Observability, and Evaluations
- the difference between a session, trace, and tool-call target
- why CityAnalyst uses fixed data and a structured response
- how a session ID connects an invocation, trace, and evaluation

Continue to [02 - Built-in On-Demand Evaluations](02-built-in-on-demand-evaluations.ipynb) to score this instrumented behavior.